# @before_agent 与 @after_agent

## LangChain @before\_agent 与 @after\_agent

before\_agent 和 after\_agent 是 Agent 级别的钩子，分别在 Agent 执行之前和完成之后各执行一次。适合做初始化、预处理、后处理和统计分析。

---

## before\_agent——Agent 开始前的准备工作

before\_agent 在 Agent 正式开始执行前运行，只执行一次。你可以在这里做输入预处理、用户信息验证、资源初始化等。

### 场景 1：输入预处理——自动修正用户输入

## 实例

from dotenv import load\_dotenv  
load\_dotenv()  
  
from langchain.agents import create\_agent  
from langchain.agents.middleware import before\_agent  
from langchain.chat\_models import init\_chat\_model  
from langchain.messages import HumanMessage  
from langchain.tools import tool  
  
@before\_agent  
def preprocess\_input(state, runtime):  
"""在 Agent 开始前处理用户输入"""  
messages = state.get("messages", \[\])  
if not messages:  
return None  
  
\# 获取用户最后一条消息  
last\_msg = messages\[-1\]  
content = str(last\_msg.content) if hasattr(last\_msg, 'content') else ""  
  
\# 自动添加礼貌用语（如果用户直接问问题）  
greetings = \["你好", "您好", "hi", "hello", "嗨"\]  
if content and not any(content.lower().startswith(g) for g in greetings):  
\# 不修改，直接返回  
pass  
  
return None  
  
@tool  
def search\_course(keyword: str) -> str:  
"""在菜鸟教程 RUNOOB 搜索课程"""  
courses = {  
"python": "Python3 基础教程（免费，30章）",  
"html": "HTML 基础教程（免费，25章）",  
}  
return courses.get(keyword.lower(), f"未找到 {keyword} 相关课程")  
  
model = init\_chat\_model("deepseek:deepseek-v4-flash", temperature=0)  
agent = create\_agent(  
model=model,  
tools=\[search\_course\],  
middleware=\[preprocess\_input\],  
system\_prompt="你是菜鸟教程 RUNOOB 的课程顾问。",  
)  
  
result = agent.invoke({  
"messages": \[HumanMessage(content="Python 课程")\]  
})  
print(f"回复: {result\['messages'\]\[-1\].content}")

### 场景 2：访问控制——权限检查

## 实例

from langchain.agents.middleware import before\_agent  
  
@before\_agent  
def access\_control(state, runtime):  
"""检查用户是否有权限使用 Agent"""  
\# 从 runtime.context 获取用户信息  
context = runtime.context  
if context is None:  
return None  
  
user\_role = context.get("user\_role", "guest")  
  
\# 访客用户只能使用有限功能  
if user\_role == "guest":  
messages = state.get("messages", \[\])  
if messages:  
last\_content = str(messages\[-1\].content)  
\# 检查是否涉及限制功能  
restricted\_keywords = \["删除", "管理", "配置", "admin"\]  
if any(kw in last\_content for kw in restricted\_keywords):  
return {  
"jump\_to": "end",  
"messages": \[HumanMessage(  
content="您当前的权限不足，无法执行此操作。请登录后重试。"  
)\]  
}  
  
return None

---

## after\_agent——Agent 完成后的处理

after\_agent 在 Agent 完成所有处理后执行（只执行一次）。你可以在这里格式化最终输出、记录统计信息、清理资源等。

### 场景 3：统计分析——记录对话数据

## 实例

from langchain.agents.middleware import after\_agent  
  
@after\_agent  
def conversation\_stats(state, runtime):  
"""统计对话信息并追加到结果中"""  
messages = state.get("messages", \[\])  
  
\# 统计数据  
model\_calls = 0  
tool\_calls = 0  
total\_chars = 0  
  
for msg in messages:  
if msg.type == "ai":  
model\_calls += 1  
if hasattr(msg, 'tool\_calls') and msg.tool\_calls:  
tool\_calls += len(msg.tool\_calls)  
if hasattr(msg, 'content') and msg.content:  
total\_chars += len(str(msg.content))  
  
\# 通过 custom stream 发送统计信息  
runtime.stream\_writer({  
"type": "stats",  
"model\_calls": model\_calls,  
"tool\_calls": tool\_calls,  
"total\_messages": len(messages),  
"total\_chars": total\_chars,  
})  
  
return None  
  
model = init\_chat\_model("deepseek:deepseek-v4-flash", temperature=0)  
agent = create\_agent(  
model=model,  
tools=\[search\_course\],  
middleware=\[after\_agent\],  
system\_prompt="你是菜鸟教程 RUNOOB 的课程顾问。",  
)  
  
\# 使用 stream\_mode=\["updates", "custom"\] 接收自定义事件  
for mode, chunk in agent.stream(  
{"messages": \[HumanMessage(content="查一下 Python 课程")\]},  
stream\_mode=\["updates", "custom"\],  
):  
if mode == "custom" and chunk.get("type") == "stats":  
print(f"统计信息: {chunk}")

运行结果：

In [ ]:
统计信息: {'type': 'stats', 'model_calls': 2, 'tool_calls': 1,
            'total_messages': 4, 'total_chars': 127}

### 场景 4：格式化输出——统一回复风格

## 实例

from langchain.agents.middleware import after\_agent  
from langchain.messages import AIMessage  
  
@after\_agent  
def format\_output(state, runtime):  
"""在结果中追加格式化的总结信息"""  
messages = state.get("messages", \[\])  
if not messages:  
return None  
  
\# 找到最后一条 AI 消息（最终回复）  
last\_ai = None  
for msg in reversed(messages):  
if msg.type == "ai" and msg.content:  
last\_ai = msg  
break  
  
if last\_ai:  
\# 统计信息  
tool\_msgs = \[m for m in messages if m.type == "tool"\]  
tool\_count = len(tool\_msgs)  
  
footer = (  
f"\\n\\n---\\n"  
f"> 本次对话共进行 {len(messages)} 条消息，"  
f"调用了 {tool\_count} 次工具。\\n"  
f"> 由菜鸟教程 RUNOOB AI 助手提供支持。"  
)  
  
\# 追加到最终回复  
return {  
"messages": \[  
AIMessage(content=last\_ai.content + footer)  
\]  
}  
  
return None  
  
model = init\_chat\_model("deepseek:deepseek-v4-flash", temperature=0)  
agent = create\_agent(  
model=model,  
tools=\[search\_course\],  
middleware=\[format\_output\],  
system\_prompt="你是菜鸟教程 RUNOOB 的课程顾问。",  
)  
  
result = agent.invoke({  
"messages": \[HumanMessage(content="Python 有什么课程？")\]  
})  
print(result\["messages"\]\[-1\].content)

运行结果：

In [ ]:
菜鸟教程 RUNOOB 中有 Python3 基础教程，共30章，完全免费，非常适合 Python 初学者入门学习。

---
> 本次对话共进行 4 条消息，调用了 1 次工具。
> 由菜鸟教程 RUNOOB AI 助手提供支持。

---

## 四个钩子的完整协作示例

## 实例

from langchain.agents.middleware import (  
before\_agent, after\_agent, before\_model, after\_model  
)  
from langchain.agents import create\_agent  
from langchain.chat\_models import init\_chat\_model  
from langchain.messages import HumanMessage  
from langchain.tools import tool  
  
\# ----- 定义所有钩子 -----  
  
@before\_agent  
def init\_session(state, runtime):  
"""开始：初始化会话"""  
print(">>> 会话开始")  
return None  
  
@before\_model  
def pre\_model\_check(state, runtime):  
"""每次模型调用前"""  
msg\_count = len(state.get("messages", \[\]))  
print(f" \[model前\] 消息数: {msg\_count}")  
return None  
  
@after\_model  
def post\_model\_check(state, runtime):  
"""每次模型调用后"""  
last = state\["messages"\]\[-1\] if state.get("messages") else None  
if last and hasattr(last, 'tool\_calls') and last.tool\_calls:  
print(f" \[model后\] 需要工具调用")  
return None  
  
@after\_agent  
def finish\_session(state, runtime):  
"""结束：清理资源"""  
total = len(state.get("messages", \[\]))  
print(f"<<< 会话结束，共 {total} 条消息")  
return None  
  
\# ----- 创建 Agent -----  
  
@tool  
def get\_weather(city: str) -> str:  
"""查询天气"""  
return f"{city}: 晴"  
  
model = init\_chat\_model("deepseek:deepseek-v4-flash", temperature=0)  
agent = create\_agent(  
model=model,  
tools=\[get\_weather\],  
middleware=\[init\_session, pre\_model\_check, post\_model\_check, finish\_session\],  
system\_prompt="你是助手。",  
)  
  
result = agent.invoke({  
"messages": \[HumanMessage(content="杭州天气？")\]  
})  
print(f"\\n最终回复: {result\['messages'\]\[-1\].content}")

运行结果：

In [ ]:
>>> 会话开始
  [model前] 消息数: 2
  [model后] 需要工具调用
  [model前] 消息数: 3
<<< 会话结束，共 4 条消息

最终回复: 杭州今天晴天，适合出行。

---

## Middleware 钩子总结

| 钩子 | 执行次数 | 何时使用 | 关键能力 |
| --- | --- | --- | --- |
| before\_agent | 1 次 | 权限检查、输入预处理、资源初始化 | 可 jump\_to="end" 提前终止 |
| before\_model | 每次循环 | 消息裁剪、内容过滤、上下文注入 | 可 jump\_to 控制流程 |
| wrap\_model\_call | 每次循环 | 重试、降级、缓存、prompt 修改 | 完全控制模型执行 |
| after\_model | 每次循环 | 响应审核、内容追加、日志 | 可替换模型输出 |
| wrap\_tool\_call | 每次工具调用 | 工具重试、缓存、参数改写 | 完全控制工具执行 |
| after\_agent | 1 次 | 输出格式化、统计分析、清理 | 最终状态修改 |